# 02e — Train Freezing Experiment (4 Konfigurasi)

Melatih **ruang desain perancangan model**: 2 backbone × 2 strategi = 4 konfigurasi.

- Backbone: `mobilenet_v3_large`, `resnet50` (Swin-Tiny dikeluarkan — reframe Juni 2026)
- Strategi: `full` (full fine-tuning) vs `freeze` (feature extraction / backbone dibekukan)

**Prasyarat:** jalankan `02_prepare_split.ipynb` dulu agar `reports/split_indices.json` tersedia.

**Output per konfigurasi** (mis. `resnet50__freeze`): `checkpoints/best_model_<config>.pth`, `reports/history_<config>.csv`, `reports/curves_<config>.png`, `reports/timing_<config>.json`, plus `reports/curves_comparison.png` dan `reports/training_summary_freezing.csv`.

## 1 · Environment detection

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)

## 2 · Mount Drive & set paths

In [ ]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah')
    WORK_ROOT  = Path('/content/work')
else:
    DRIVE_ROOT = Path('..').resolve()
    WORK_ROOT  = DRIVE_ROOT

WORK_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'reports').mkdir(parents=True, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)
print('WORK_ROOT  =', WORK_ROOT)

## 3 · Unzip / locate dataset

In [ ]:
import zipfile

DATA_ZIP  = DRIVE_ROOT / 'Dataset_Cropped.zip'
DATA_ROOT = WORK_ROOT

if not (DATA_ROOT / 'defect').exists():
    if IN_COLAB:
        if not DATA_ZIP.exists():
            raise FileNotFoundError(f'Dataset zip not found. Upload Dataset_Cropped.zip to: {DATA_ZIP}')
        print(f'Extracting {DATA_ZIP} -> {WORK_ROOT} ...')
        with zipfile.ZipFile(DATA_ZIP) as zf:
            zf.extractall(WORK_ROOT)
        print('Done.')
    else:
        raise FileNotFoundError(f'Dataset folder not found at: {DATA_ROOT}. Run 01_auto_crop.py first.')
else:
    print(f'Dataset already present at {DATA_ROOT} -- skipping extract.')

for c in ('defect', 'non_defect'):
    folder = DATA_ROOT / c
    if not folder.exists():
        raise FileNotFoundError(f'Expected class folder not found: {folder}')
    n = len(list(folder.glob('*.jpg')))
    print(f'  {c:>10}: {n:>5} images')

## 4 · Import utils.py

In [ ]:
import importlib, shutil, sys

src_utils = DRIVE_ROOT / 'scripts' / 'utils.py'
if src_utils.exists():
    shutil.copy(src_utils, '/content/utils.py' if IN_COLAB else 'utils.py')
    sys.path.insert(0, '/content' if IN_COLAB else '.')
else:
    print(f'[warn] {src_utils} not found -- paste utils.py manually.')

import utils
importlib.reload(utils)
from utils import (
    NUM_CLASSES, IMG_SIZE,
    BACKBONES, STRATEGIES, CONFIG_NAMES, make_config_name,
    seed_everything, get_transforms, TransformSubset,
    build_model, count_trainable_params, count_total_params,
    freeze_backbone, train_one_model, save_comparison_curves,
)
print('Konfigurasi yang akan dilatih:', CONFIG_NAMES)

## 5 · Reproducibility, device & hyperparameters

In [ ]:
import torch

SEED = 42
seed_everything(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
print('Device:', DEVICE, '| AMP:', USE_AMP)
if DEVICE.type == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))

In [ ]:
HPARAMS = {
    'batch_size'   : 64,
    'num_workers'  : 4,
    'epochs'       : 25,
    'learning_rate': 1e-4,
    'weight_decay' : 1e-4,
    'patience'     : 5,
    'split_ratios' : (0.8, 0.1, 0.1),
    'seed'         : 42,
    'img_size'     : 224,
}
HPARAMS

## 6 · Load split & build DataLoaders (sama untuk semua konfigurasi)

In [ ]:
import json
from torch.utils.data import DataLoader, Subset
from torchvision import datasets

split_path = DRIVE_ROOT / 'reports' / 'split_indices.json'
if not split_path.exists():
    raise FileNotFoundError('Run 02_prepare_split.ipynb first')

with open(split_path) as f:
    split = json.load(f)

full_dataset = datasets.ImageFolder(root=str(DATA_ROOT), transform=None)
print('Classes:', full_dataset.class_to_idx)
print('Total images:', len(full_dataset))

saved_basenames = [Path(p).name for p in split['samples']]
current_basenames = [Path(s).name for s, _ in full_dataset.samples]
if saved_basenames != current_basenames:
    raise ValueError('Dataset basenames do not match split_indices.json. Re-run 02_prepare_split.ipynb.')
print('Basename validation passed')

In [ ]:
train_subset = Subset(full_dataset, split['train_indices'])
val_subset   = Subset(full_dataset, split['val_indices'])

train_ds = TransformSubset(train_subset, get_transforms(train=True))
val_ds   = TransformSubset(val_subset,   get_transforms(train=False))

seed_everything(HPARAMS['seed'])
common = dict(batch_size=HPARAMS['batch_size'], num_workers=HPARAMS['num_workers'],
              pin_memory=DEVICE.type == 'cuda')
train_loader = DataLoader(train_ds, shuffle=True,  drop_last=False, **common)
val_loader   = DataLoader(val_ds,   shuffle=False, drop_last=False, **common)
print(f'Train: {len(train_subset)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_subset)} samples, {len(val_loader)} batches')

## 7 · Latih 4 konfigurasi (backbone × strategi)

Tiap konfigurasi memakai `train_one_model` dari `utils.py`. Mode `freeze` membekukan backbone (feature extraction); mode `full` melatih seluruh parameter (full fine-tuning). Hyperparameter seragam agar perbandingan adil.

In [ ]:
results = {}
for backbone in BACKBONES:
    for strategy in STRATEGIES:
        cfg = make_config_name(backbone, strategy)
        results[cfg] = train_one_model(
            model_name=backbone,
            train_loader=train_loader,
            val_loader=val_loader,
            full_dataset=full_dataset,
            hparams=HPARAMS,
            device=DEVICE,
            use_amp=USE_AMP,
            ckpt_dir=DRIVE_ROOT / 'checkpoints',
            report_dir=DRIVE_ROOT / 'reports',
            freeze=(strategy == 'freeze'),
            config_name=cfg,
        )

print('\nSelesai melatih', len(results), 'konfigurasi.')

## 8 · Kurva perbandingan + ringkasan

In [ ]:
import pandas as pd

save_comparison_curves(results, DRIVE_ROOT / 'reports', order=CONFIG_NAMES)

summary = pd.DataFrame([
    {
        'config'      : r['config_name'],
        'backbone'    : r['model_name'],
        'strategy'    : r['strategy'],
        'best_epoch'  : r['best_epoch'],
        'best_val_acc': round(float(r['best_val_acc']), 4),
        'train_secs'  : round(float(r['train_secs']), 1),
    }
    for r in results.values()
])
summary.to_csv(DRIVE_ROOT / 'reports' / 'training_summary_freezing.csv', index=False)
print(summary.to_string(index=False))

✅ **Selesai.** 4 checkpoint + history/curves/timing tersimpan di `reports/` & `checkpoints/`.

Lanjut: jalankan `05_evaluate_models.ipynb` (sudah diperbarui untuk mengevaluasi 4 konfigurasi).